# CIM SoC 全功能板上验证

使用 `cim_driver.py`（CIMDriver / CIMModel）+ `golden_model.py` 进行：
1. AXI 连通性测试
2. 随机 MLP（784→128→10）bit-exact 逐层验证
3. 批量推理正确率测试

**前置条件**：将以下文件拷贝到 PYNQ 板同一目录下：
- `cim_soc.bit` + `cim_soc.hwh`（bitstream）
- `cim_driver.py`、`golden_model.py`（SW 驱动 + golden model）
- 本 notebook

In [ ]:
# ====================================================================
# 1. 导入
# ====================================================================
import numpy as np
from cim_driver import (
    CIMDriver, CIMModel, _make_run_id,
    weight_to_chunks, bias_to_u32,
)
import golden_model as gm

In [ ]:
# ====================================================================
# 2. 加载 bitstream + 创建驱动
# ====================================================================
drv = CIMDriver("cim_soc.bit")
print("Overlay loaded, driver ready.")

In [ ]:
# ====================================================================
# 3. AXI 连通性测试
# ====================================================================
from cim_driver import _BASE, _MMIO_SIZE
from pynq import MMIO

mmio = MMIO(_BASE, _MMIO_SIZE)
CSR_IN_DIM = 0x010

print("=== AXI Connectivity Test ===")
drv.soft_reset()
mmio.write(CSR_IN_DIM, 784)
rb = mmio.read(CSR_IN_DIM)
print(f"  IN_DIM write=784, readback={rb}  {'PASS' if rb == 784 else 'FAIL'}")
assert rb == 784, "AXI read/write failed! Check bitstream and address mapping."

mmio.write(CSR_IN_DIM, 128)
rb = mmio.read(CSR_IN_DIM)
print(f"  IN_DIM write=128, readback={rb}  {'PASS' if rb == 128 else 'FAIL'}")
assert rb == 128
print("AXI OK.")

In [ ]:
# ====================================================================
# 4. 生成随机 MLP 测试数据 (seed=42, 与 golden_model --mnist-e2e 一致)
# ====================================================================
np.random.seed(42)
w1 = np.random.randint(-128, 127, (128, 784), dtype=np.int8)
b1 = np.random.randint(-5000, 5000, 128, dtype=np.int32)
w2 = np.random.randint(-128, 127, (10, 128), dtype=np.int8)
b2 = np.random.randint(-5000, 5000, 10, dtype=np.int32)
img = np.random.randint(0, 255, 784, dtype=np.uint8)

# --- Calibrate requant params ---
def calibrate_requant(acc_values, shift=16):
    """Compute (mult, shift) that maps max(abs(acc)) -> 127."""
    max_abs = max(abs(int(acc_values.max())), abs(int(acc_values.min())), 1)
    mult = max(1, int(round(127.0 / max_abs * (1 << shift))))
    return mult, shift

# FC1 calibration
x_eff1 = np.clip(img.astype(np.int32) - (-128), 0, 511).astype(np.int32)
acc1 = w1.astype(np.int32) @ x_eff1 + b1.astype(np.int32)
fc1_mult, fc1_shift = calibrate_requant(np.maximum(acc1, 0))

# FC1 golden -> FC2 calibration
fc1_golden = gm.infer_layer(img, w1, b1, -128, fc1_mult, fc1_shift, "relu")
fc2_in = fc1_golden["output"].view(np.uint8)
x_eff2 = np.clip(fc2_in.astype(np.int32), 0, 511).astype(np.int32)
acc2 = w2.astype(np.int32) @ x_eff2 + b2.astype(np.int32)
fc2_mult, fc2_shift = calibrate_requant(acc2)

fc2_golden = gm.infer_layer(fc2_in, w2, b2, 0, fc2_mult, fc2_shift, "none")
expected_class = fc2_golden["pred_class"]

print(f"FC1: mult={fc1_mult}, shift={fc1_shift}")
print(f"FC2: mult={fc2_mult}, shift={fc2_shift}")
print(f"Golden FC1 output (first 10): {fc1_golden['output'][:10]}")
print(f"Golden FC2 output: {fc2_golden['output']}")
print(f"Expected class: {expected_class}")

In [ ]:
# ====================================================================
# 5. 单图 MLP 推理 + bit-exact 逐层验证 (Step 6 Phase 1)
# ====================================================================
model = CIMModel(drv)
model.add_fc(784, 128, weight_to_chunks(w1), bias_to_u32(b1),
             zp=-128, mult=fc1_mult, shift=fc1_shift, relu=True,
             weight_int8=w1, bias_int32=b1)
model.add_fc(128, 10, weight_to_chunks(w2), bias_to_u32(b2),
             zp=0, mult=fc2_mult, shift=fc2_shift, relu=False,
             weight_int8=w2, bias_int32=b2)

print("=== Single Image MLP Inference (verify=True) ===")
pred, logits = model.predict(img, verbose=True, verify=True)

print(f"\nHW predicted class: {pred}")
print(f"Golden expected:    {expected_class}")
print(f"Argmax: {'MATCH' if pred == expected_class else 'MISMATCH'}")
print(f"Logits: {list(np.asarray(logits, dtype=np.int8))}")

In [ ]:
# ====================================================================
# 6. 批量推理：N 张随机图，统计正确率 + 平均 cycle
# ====================================================================
import time

N_IMAGES = 20
correct = 0
total_cycles_all = 0

print(f"=== Batch Test: {N_IMAGES} random images ===")
t0 = time.time()

for n in range(N_IMAGES):
    np.random.seed(1000 + n)
    test_img = np.random.randint(0, 255, 784, dtype=np.uint8)

    # Golden reference (software)
    fc1_g = gm.infer_layer(test_img, w1, b1, -128, fc1_mult, fc1_shift, "relu")
    fc2_in_g = fc1_g["output"].view(np.uint8)
    fc2_g = gm.infer_layer(fc2_in_g, w2, b2, 0, fc2_mult, fc2_shift, "none")
    golden_class = fc2_g["pred_class"]

    # Hardware inference (reuse same model)
    pred, logits = model.predict(test_img, verbose=False)

    hw_logits = np.asarray(logits, dtype=np.int8)
    golden_logits = fc2_g["output"]
    bit_exact = np.array_equal(hw_logits, golden_logits)

    if pred == golden_class:
        correct += 1
    if not bit_exact:
        diffs = np.where(hw_logits != golden_logits)[0]
        print(f"  img #{n}: pred={pred} golden={golden_class}  "
              f"LOGIT MISMATCH at {len(diffs)} indices")

elapsed = time.time() - t0
print(f"\nResults: {correct}/{N_IMAGES} class match "
      f"({100*correct/N_IMAGES:.1f}%)")
print(f"Wall time: {elapsed:.2f}s ({elapsed/N_IMAGES*1000:.1f} ms/image)")